# RQ2 Analysis

This notebook is used to answer RQ2: **Does a continuous adaptive control architecture produce a proportional environmental response that reflects the user's physiological stress trajectory throughout the session?**

Each adaptive session CSV includes two columns for every heartbeat cycle:
- `difference`: how far RR-interval is from baseline
- `reaction_score`: the PID controller's output

We calculate the Pearson correlation coefficient (r) between these two columns, per participant. r tells us how closely the two move together. If r is close to +1 or -1, it means that there is a strong relationship. If r is closer to 0, it means there is a weak relationship or no relationship.

**Note:** This notebook requires the raw adaptive session CSVs (`01_adaptive.csv`-`11_adaptive.csv`) and the CSV containing the SAM scores, file links and condition order (`participant_info.csv`) which are not included in the supporting material submission, consistent with the data access terms  agreed with participants (see Reflective Essay, Section 6). This notebook's cells retain their originally executed output below, so results remain visible and verifiable by inspection without requiring the underlying raw data.

In [1]:
from scipy.stats import pearsonr
import csv

## 1. Linking files to participants

In [2]:
def load_participant_files(filename="participant_info.csv"):
    """
    Loads a dictionary mapping participant ID to adaptive-file name from participant_info.csv.
    This file is not included in the supporting material submission, 
    consistent with the data access terms agreed with participants (see Reflective Essay, Section 6).
    """
    participant_files = {}
    with open(filename, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            participant_files[row["participant_id"]] = row["adaptive_file"]
    return participant_files


participant_files = load_participant_files()

## 2. Reading file and getting reaction scores and differences

In [3]:
def read_difference_and_reaction_score(filename):
    """Opens one adaptive session CSV and pulls out differences and reaction scores. Returns two parallel lists, ensuring that the values at any given index correspond to the 
    same heartbeat cycle."""
    differences = []
    reaction_scores = []

    with open(filename, "r") as adaptive_file:
        reader = csv.DictReader(adaptive_file) # Tells program to read first row as headers
        for row in reader:
            try:
                differences.append(float(row["difference"]))
                reaction_scores.append(float(row["reaction_score"]))
            except (ValueError, KeyError):
                pass  # skips any broken/missing rows

    return differences, reaction_scores

## 3. Running correlation for every participant

In [4]:
results = []

for participant_id, filename in participant_files.items():
    differences, reaction_scores = read_difference_and_reaction_score(filename)

    r_value, p_value = pearsonr(differences, reaction_scores) # pearsonr outputs r and p-value

    results.append({
        "id": participant_id,
        "file": filename,
        "n_heartbeats": len(differences),
        "r": r_value,
        "p": p_value,
    })

## 4. Printing a table and summary

In [5]:
print("INDIVIDUAL RESULTS:")
print()

all_r_values = []

for row in results:
    r_rounded = round(row['r'], 3)
    p_rounded = round(row['p'], 4)

    print(f"Participant: {row['id']} | File: {row['file']} | Beats: {row['n_heartbeats']} | r: {r_rounded} | p: {p_rounded}")
    
    all_r_values.append(row["r"]) # Appending each participant's r score into an overall list to calculate a group summary

mean_r = sum(all_r_values) / len(all_r_values)
min_r = min(all_r_values)
max_r = max(all_r_values)

# Printing final summary
print("\nGROUP SUMMARY:")
print()
print(f"Mean r = {round(mean_r, 3)}")
print(f"Range = {round(min_r, 3)} to {round(max_r, 3)}")

INDIVIDUAL RESULTS:

Participant: P01 | File: 01_adaptive.csv | Beats: 241 | r: 0.967 | p: 0.0
Participant: P02 | File: 02_adaptive.csv | Beats: 247 | r: 0.973 | p: 0.0
Participant: P03 | File: 03_adaptive.csv | Beats: 234 | r: 0.978 | p: 0.0
Participant: P04 | File: 04_adaptive.csv | Beats: 185 | r: 0.988 | p: 0.0
Participant: P05 | File: 05_adaptive.csv | Beats: 194 | r: 0.993 | p: 0.0
Participant: P06 | File: 06_adaptive.csv | Beats: 227 | r: 0.978 | p: 0.0
Participant: P07 | File: 07_adaptive.csv | Beats: 288 | r: 0.955 | p: 0.0
Participant: P08 | File: 08_adaptive.csv | Beats: 253 | r: 0.976 | p: 0.0
Participant: P09 | File: 09_adaptive.csv | Beats: 224 | r: 0.993 | p: 0.0
Participant: P10 | File: 10_adaptive.csv | Beats: 288 | r: 0.983 | p: 0.0
Participant: P11 | File: 11_adaptive.csv | Beats: 244 | r: 0.952 | p: 0.0

GROUP SUMMARY:

Mean r = 0.976
Range = 0.952 to 0.993


Note: because `reaction_score` is computed from `difference` via the PID formula, a strong correlation here shouldn't be surprising, rather a confirmation that the code is working as intended by turning heart rate deviation smoothly into an output.